# Как проверяем качество: валидация

**Что есть в данных.**

- **Корпус** — файл `benchmark_items.parquet`, 189 212 объявлений услуг. Только среди них выбираем 50 ответов на тестовый запрос.
- **Тестовые запросы** — файл `benchmark_queries.parquet`, 2 452 запроса с номерами `query_id`. Ответов к ним у нас нет, они есть только у организаторов. Мы отправляем организаторам по 50 объявлений на каждый тестовый запрос и получаем в ответ качество решения — Recall@50 (определение в разделе 1). Таблица с результатами участников называется лидербордом.
- **Train** — файл `train.parquet`, 497 673 строки. Каждая строка — один выбор пользователя: поля поиска и одно объявление, которое он выбрал. Поля поиска — это текст, локация (номер города или региона), фильтры, категория (искали в разделе «Услуги» или по всему Авито) и отметка о том, искал ли пользователь с доставкой. Фильтры — то, что пользователь уточнил на сайте, например «Вид услуги Красота, здоровье» или «Кто оказывает услуги Частный исполнитель». Номера запроса у строк нет.

Только в 6.6% строк train выбранное объявление есть в корпусе (раздел 6). Остальные выбранные объявления в корпус не входят, и найти их поиском по корпусу нельзя. Такие строки не годятся для проверки качества, но по ним можно считать доли выборов по локациям и доучивать эмбеддер (что это, сказано ниже).

**Как устроено решение.** Решение ищет в два шага. Сначала ретривер отбирает для запроса около 600 кандидатов из корпуса (`ranking/dense.ipynb`). Ретривер состоит из двух поисков. BM25 ищет объявления, в которых есть слова запроса, и выше ставит те, где совпало больше редких слов. Поиск по смыслу находит объявления, близкие к запросу по смыслу, даже если слова не совпадают. Для него нужен эмбеддер — нейросеть, которая превращает текст в вектор так, что у похожих по смыслу текстов векторы близки (близость меряем косинусом угла между векторами). Готовую нейросеть мы доучиваем на парах «запрос → выбранное объявление» из train. Затем реранкер — модель LightGBM — по подсказкам о каждом кандидате заново упорядочивает кандидатов и берёт первые 50.

Одна из подсказок реранкера — доли выборов по локациям. Для каждой локации поиска по train считаем, в каких локациях и как часто выбирали объявления. Например, при поиске по «Москве и области» 61% выборов приходится на саму Москву, остальное — на города области (`ingestion/insights.ipynb`). Чем чаще при таком поиске выбирали объявления из локации кандидата, тем выше его стоит поставить.

**Что в этом ноутбуке.** Чтобы проверять качество у себя, из train собираем проверочные запросы с известными ответами — валидацию. Она полезна, только если её запросы похожи на тестовые и поиск по ним идёт по тому же корпусу. Здесь показано, как она собрана и насколько похожа на тест.

Здесь же train делится на три части: валидация, запросы для обучения реранкера и остаток — обучение ретривера, по которому считаются доли выборов по локациям и доучивается эмбеддер (раздел 5). Делит `scripts/validation/split.py`, код — `avito.validation.split`.

In [1]:
import polars as pl

from avito.columns import QUERY_COLUMNS
from avito.ingestion import load as ingested
from avito.processing import load as processed
from avito.validation.load import load_ranker_queries, load_retriever_fold, load_train_fold, load_val_queries
from avito.validation.split import AUTO_FILTER, bench_seen_profile, build_tuples, check, check_ranker, popularity_bin

pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(80)
pl.Config.set_tbl_width_chars(220);

Загружаем train, тестовые запросы, номера объявлений корпуса и список локаций, в которых в корпусе есть хотя бы одно объявление.

In [2]:
train = processed.load_train()
bench = processed.load_bench_queries()
corpus_item_ids = processed.load_corpus(["item_id"])
corpus_locations = ingested.load_corpus(["item_location_id"]).unique().rename({"item_location_id": "search_location_id"})

## 1. Как из строк train получаются запросы

В тесте у каждого запроса есть номер, и для проверки нам тоже нужны запросы со списком правильных ответов. А в train есть только отдельные строки «поля поиска → выбранное объявление». Поэтому склеиваем в один запрос все строки train с одинаковыми полями поиска — текстом, локацией, фильтрами, категорией и отметкой о поиске с доставкой — и сами даём ему номер. Правильные ответы такого запроса — все объявления, выбранные в его строках. Один и тот же текст, набранный в разных городах или с разными фильтрами, — это разные запросы.

Сколько запросов получается:

In [3]:
tuples = build_tuples(train, corpus_item_ids)
chosen_items = train.group_by(QUERY_COLUMNS).agg(pl.col("item_id").n_unique().alias("items"))

pl.DataFrame({
    "показатель": [
        "строк train",
        "разных запросов (разных наборов полей поиска)",
        "запросов, склеенных из нескольких строк",
        "запросов, в строках которых выбирали разные объявления",
    ],
    "значение": [train.height, tuples.height, (tuples["row_id"].list.len() > 1).sum(), (chosen_items["items"] > 1).sum()],
})

показатель,значение
str,i64
"""строк train""",497673
"""разных запросов (разных наборов полей поиска)""",354463
"""запросов, склеенных из нескольких строк""",58166
"""запросов, в строках которых выбирали разные объявления""",53166


Из 497 673 строк получается 354 463 запроса. Большинство запросов — одна строка. 58 тысяч запросов склеены из нескольких строк, и почти во всех из них выбирали разные объявления.

Пример: две строки train с одинаковыми полями поиска, в которых выбрали разные объявления. Они станут одним запросом с двумя правильными ответами.

In [4]:
example = tuples.filter(
    pl.col("all_in_corpus") & (pl.col("row_id").list.len() == 2) & (pl.col("relevant_item_ids").list.len() == 2)
).sort("search_query").row(0, named=True)
titles = ingested.load_items(["item_id", "item_title_raw"])

train.filter(pl.col("row_id").is_in(example["row_id"])).join(titles, on="item_id", maintain_order="left").select(
    pl.col("search_query").alias("текст"),
    pl.col("search_location_id").alias("локация"),
    pl.col("search_infm_params_text").alias("фильтры"),
    pl.col("item_id").alias("выбранное объявление"),
    pl.col("item_title_raw").alias("его заголовок"),
)

текст,локация,фильтры,выбранное объявление,его заголовок
str,i64,str,str,str
"""3d печать""",625390,"""Вид услуги Оборудование, производство""","""186b0e8fc56449b9""","""3D Печать. 3D Моделирование. 3D Сканирование"""
"""3d печать""",625390,"""Вид услуги Оборудование, производство""","""33530b6f8a1e84f2""","""3Д печать, 3Д моделирование, Реверс инжениринг"""


**Recall@50** запроса — доля его правильных объявлений, попавших в 50 ответов. Если у запроса два правильных объявления, а среди ответов нашлось одно, Recall@50 равен 0.5. Качество решения — среднее Recall@50 по всем запросам; так же считается и качество на лидерборде.

В условии задачи сказано, что подходящих объявлений у тестового запроса «обычно одно-два, иногда больше». Наша склейка тоже допускает несколько правильных объявлений у запроса. Но в валидации таких запросов мало: у 4 977 из 5 032 запросов правильное объявление одно, у остальных 55 — два или три (раздел 4 объясняет, почему так вышло). Это одно из отличий валидации от теста (раздел 6).

## 2. Какими должны быть проверочные запросы

Сравниваем тестовые запросы со всеми строками train и с разными текстами train, где каждый текст взят один раз. В колонке «размер» — число строк, текстов или запросов соответственно. Сравниваем среднее число слов в тексте, долю однословных текстов, долю без фильтров и долю с фильтром «Автосервис, аренда». Этот фильтр входит в самые частые наборы фильтров train (`ingestion/raw.ipynb`), поэтому проверяем, встречается ли он в тесте.

In [5]:
def profile(name: str, df: pl.DataFrame) -> dict:
    words = df["query_stems"].str.split(" ").list.len()
    return {
        "что": name,
        "размер": df.height,
        "слов в тексте в среднем": words.mean(),
        "доля однословных текстов": (words == 1).mean(),
        "доля без фильтров": (df["search_infm_params_text"] == "").mean(),
        "доля с фильтром «Автосервис, аренда»": df["search_infm_params_text"].str.contains(AUTO_FILTER, literal=True).mean(),
    }


pl.DataFrame([
    profile("строки train", train),
    profile("тексты train, каждый по одному разу", train.unique("search_query", keep="first")),
    profile("тестовые запросы", bench),
])

что,размер,слов в тексте в среднем,доля однословных текстов,доля без фильтров,"доля с фильтром «Автосервис, аренда»"
str,i64,f64,f64,f64,f64
"""строки train""",497673,2.14607,0.196816,0.329532,0.124417
"""тексты train, каждый по одному разу""",74529,2.752311,0.045217,0.474366,0.108361
"""тестовые запросы""",2452,2.798124,0.046493,0.630506,0.0


Тестовые запросы похожи на тексты train, взятые по одному разу, а не на все строки train. В строках train много коротких частых текстов вроде «маникюр»: каждая пятая строка состоит из одного слова. Среди тестовых запросов однословных меньше 5%, и столько же среди текстов train, взятых по одному разу. Поэтому в валидацию тоже берём каждый текст один раз.

По доле запросов без фильтров похожести нет: 47% у текстов train против 63% в тесте. К этому отличию вернёмся в разделах 4 и 6.

Фильтра «Автосервис, аренда» в тесте нет вообще, а в train он есть примерно в каждой восьмой строке. Такие запросы в валидацию не берём.

Ещё одна важная характеристика теста — какая доля тестовых текстов уже встречалась в train (с той же или с другой локацией и фильтрами). Такие тексты дальше называем **знакомыми**, остальные — **новыми**. По знакомым текстам решение может опереться на то, что выбирали раньше, по новым — нет. Считаем эту долю и заодно проверяем, что все тексты тестовых запросов разные:

In [6]:
train_texts = train.select("search_query").unique()

pl.DataFrame({
    "показатель": ["доля тестовых запросов, текст которых встречается в train", "разных текстов среди 2 452 тестовых запросов"],
    "значение": [bench.join(train_texts, on="search_query", how="semi").height / bench.height, float(bench["search_query"].n_unique())],
})

показатель,значение
str,f64
"""доля тестовых запросов, текст которых встречается в train""",0.369902
"""разных текстов среди 2 452 тестовых запросов""",2452.0


Все 2 452 тестовых текста разные. 37% из них знакомые, остальные 63% — новые. В валидации должна быть та же доля.

## 3. Из каких запросов можно собрать проверку

Проверочный запрос должен искать среди тех же 189 212 объявлений корпуса, что и тестовый. Если выбранное по нему объявление не лежит в корпусе, найти его невозможно. Поэтому берём только запросы, у которых все выбранные объявления лежат в корпусе, — у тестовых запросов это так по условию задачи. Такие запросы дальше называем подходящими. Считаем, сколько их в train:

In [7]:
pl.DataFrame({
    "показатель": ["разных запросов в train", "из них все выбранные объявления лежат в корпусе"],
    "значение": [tuples.height, tuples.filter(pl.col("all_in_corpus")).height],
})

показатель,значение
str,i64
"""разных запросов в train""",354463
"""из них все выбранные объявления лежат в корпусе""",18436


Подходит только 5% запросов train: большая часть выбранных объявлений в корпус не попала. Из подходящих убираем запросы с фильтром «Автосервис, аренда», оставляем по одному запросу на каждый текст и смотрим, какая доля этих текстов знакомая, то есть встречается в train ещё в каком-то запросе:

In [8]:
candidates = tuples.filter(pl.col("all_in_corpus"))
candidates = candidates.filter(~pl.col("search_infm_params_text").str.contains(AUTO_FILTER, literal=True))
one_per_text = candidates.unique("search_query", keep="first")

pl.DataFrame({
    "показатель": ["разных текстов среди подходящих запросов", "доля текстов, которые встречаются в train ещё в каком-то запросе"],
    "значение": [float(one_per_text.height), one_per_text["seen_text"].mean()],
})

показатель,значение
str,f64
"""разных текстов среди подходящих запросов""",10542.0
"""доля текстов, которые встречаются в train ещё в каком-то запросе""",0.699203


Среди подходящих запросов 70% текстов знакомые, а в тесте — 37%. Разберёмся, откуда такой перекос. Для знакомых и новых текстов train считаем, какая доля их запросов подходит и у какой доли текстов есть хотя бы один подходящий запрос. Запросы с фильтром «Автосервис, аренда», как и выше, не считаем.

In [9]:
no_auto = tuples.filter(~pl.col("search_infm_params_text").str.contains(AUTO_FILTER, literal=True))
texts = no_auto.group_by("search_query").agg(pl.col("seen_text").first(), pl.col("all_in_corpus").any().alias("has_candidate"))

pl.DataFrame([
    {
        "тексты": name,
        "запросов в train": no_auto.filter(pl.col("seen_text") == seen).height,
        "доля подходящих запросов": no_auto.filter(pl.col("seen_text") == seen)["all_in_corpus"].mean(),
        "разных текстов": texts.filter(pl.col("seen_text") == seen).height,
        "доля текстов, у которых есть подходящий запрос": texts.filter(pl.col("seen_text") == seen)["has_candidate"].mean(),
        "текстов, у которых есть подходящий запрос": texts.filter((pl.col("seen_text") == seen) & pl.col("has_candidate")).height,
    }
    for name, seen in [("знакомые", True), ("новые", False)]
])

тексты,запросов в train,доля подходящих запросов,разных текстов,"доля текстов, у которых есть подходящий запрос","текстов, у которых есть подходящий запрос"
str,i64,f64,i64,f64,i64
"""знакомые""",267863,0.052837,26178,0.281572,7371
"""новые""",41801,0.075859,41801,0.075859,3171


Дело не в том, что по знакомым текстам чаще выбирали объявления из корпуса: в расчёте на один запрос подходящих у них даже меньше — 5.3% против 7.6% у новых текстов. Дело в числе запросов. У знакомого текста в train много запросов, и хотя бы один из них с большей вероятностью подходит: так бывает у 28% знакомых текстов и только у 7.6% новых. Поэтому среди подходящих запросов знакомых текстов гораздо больше, чем в тесте.

Чтобы доля совпала с тестом, берём все 3 171 новый текст и добавляем знакомые, пока их доля не станет 37%. Для этого знакомых нужно около 1 860 (3 171 × 37 / 63), и всего получается около 5 030 запросов. Взять все 10 542 подходящих текста нельзя: знакомых тогда было бы 70%, а на знакомых текстах качество выше (`ranking/dense.ipynb`, раздел 8), так что оценка на валидации была бы завышена.

Важно не только сколько знакомых текстов, но и насколько они популярны. Если набрать в валидацию одни частые тексты, любая подсказка модели вида «что выбирали по этому тексту раньше» будет на валидации работать лучше, чем на тесте, и мы переоценим качество решения. Поэтому знакомые тексты делим на группы по тому, в скольких разных запросах train (с разными локациями или фильтрами) встречается этот текст: в 1, в 2–3, в 4–10, в 11–50 или больше чем в 50. Знакомые тексты валидации набираем в тех же долях, что в тесте. Это честно: тексты тестовых запросов даны в условии задачи, а их правильных ответов мы не видим и не используем.

Доли групп среди знакомых тестовых текстов:

In [10]:
BIN_NAMES = {"(-inf, 1]": "1", "(1, 3]": "2–3", "(3, 10]": "4–10", "(10, 50]": "11–50", "(50, inf]": "больше 50"}
bins = pl.DataFrame({"popularity_bin": list(BIN_NAMES), "в скольких запросах train встречается текст": list(BIN_NAMES.values())})

seen_share, bench_bins = bench_seen_profile(bench, tuples)

bins.join(bench_bins, on="popularity_bin", maintain_order="left").select(
    "в скольких запросах train встречается текст",
    pl.col("bench_texts").alias("знакомых тестовых текстов"),
    pl.col("share").alias("доля"),
)

в скольких запросах train встречается текст,знакомых тестовых текстов,доля
str,u32,f64
"""1""",253,0.278942
"""2–3""",243,0.267916
"""4–10""",255,0.281147
"""11–50""",117,0.128997
"""больше 50""",39,0.042999


## 4. Итоговая валидация

Собирает её `scripts/validation/split.py`, здесь загружаем результат. Отбор такой. Подходящие запросы без фильтра «Автосервис, аренда» перемешиваются в случайном порядке, и у каждого текста остаётся первый запрос после перемешивания, то есть случайный. Новые тексты берутся все. Знакомые берутся случайно внутри каждой группы популярности, в долях теста. Начальное значение генератора случайных чисел зафиксировано, поэтому при повторном запуске получается та же валидация.

Все строки проверочных запросов удаляются из train, чтобы доли выборов по локациям, эмбеддер и реранкер, качество которого мы проверяем, не видели их ответов. Итоговый реранкер для ответа потом учится и на валидации (раздел 5).

In [11]:
val = load_val_queries()
train_fold = load_train_fold()

pl.DataFrame({
    "показатель": ["проверочных запросов", "строк train после удаления проверочных запросов", "удалено строк"],
    "значение": [val.height, train_fold.height, train.height - train_fold.height],
})

показатель,значение
str,i64
"""проверочных запросов""",5032
"""строк train после удаления проверочных запросов""",492276
"""удалено строк""",5397


Как выглядят проверочные запросы — первые пять:

In [12]:
val.select(
    pl.col("search_query").alias("текст"),
    pl.col("search_location_id").alias("локация"),
    pl.col("search_infm_params_text").alias("фильтры"),
    pl.col("relevant_item_ids").alias("правильные объявления"),
    pl.col("seen_text").alias("текст встречается в train"),
).head(5)

текст,локация,фильтры,правильные объявления,текст встречается в train
str,i64,str,list[str],bool
"""грузоперевозки 24 часа""",637640,"""""","[""b62a44f441ab60a0""]",false
"""реклама на заднем стекле""",621565,"""""","[""14bb165b405e798c""]",false
"""канекалон плетение""",640860,"""Кто оказывает услуги Частный исполнитель""","[""27005223c7cec89e""]",false
"""ремонт беспроводных наушников""",662810,"""""","[""701d54d78f79413e""]",false
"""двойник""",653240,"""""","[""359192adbce8045f""]",false


Сравниваем валидацию с тестом по тем же показателям, что в разделе 2, и ещё по трём: доле знакомых текстов, доле запросов, в локации поиска которых в корпусе нет ни одного объявления, и числу запросов по всему Авито.

Для валидации «текст встречается в train» считается по train без проверочных запросов, так же как для теста — по всему train. Иначе любой проверочный текст считался бы знакомым: он есть в train хотя бы в строках самого проверочного запроса.

Часть тестовых запросов искала не в разделе «Услуги», а по всему Авито (подробнее — раздел 6). Фильтров у таких запросов нет, поэтому отдельной строкой показываем тест без них: так видно, насколько они завышают долю тестовых запросов без фильтров.

In [13]:
fold_texts = train_fold.select("search_query").unique()


def full_profile(name: str, df: pl.DataFrame, known_texts: pl.DataFrame) -> dict:
    result = profile(name, df)
    result["доля текстов, встречающихся в train"] = df.join(known_texts, on="search_query", how="semi").height / df.height
    result["доля запросов, в локации поиска которых нет объявлений корпуса"] = df.join(corpus_locations, on="search_location_id", how="anti").height / df.height
    result["запросов по всему Авито"] = (df["search_category"] == 0).sum()
    return result


pl.DataFrame([
    full_profile("тестовые запросы", bench, train_texts),
    full_profile("тестовые запросы, кроме поиска по всему Авито", bench.filter(pl.col("search_category") != 0), train_texts),
    full_profile("валидация", val, fold_texts),
])

что,размер,слов в тексте в среднем,доля однословных текстов,доля без фильтров,"доля с фильтром «Автосервис, аренда»","доля текстов, встречающихся в train","доля запросов, в локации поиска которых нет объявлений корпуса",запросов по всему Авито
str,i64,f64,f64,f64,f64,f64,f64,i64
"""тестовые запросы""",2452,2.798124,0.046493,0.630506,0.0,0.369902,0.174144,222
"""тестовые запросы, кроме поиска по всему Авито""",2230,2.825561,0.042601,0.593722,0.0,0.396413,0.16278,0
"""валидация""",5032,2.759738,0.038156,0.52345,0.0,0.369833,0.175278,1


Валидация совпадает с тестом по длине запросов, доле знакомых текстов (37%) и доле запросов, в локации поиска которых в корпусе нет ни одного объявления (около 17%). Как показано в `ingestion/insights.ipynb`, это поиски по региону («Москва и область») или по всей стране: объявления привязаны к конкретным городам, поэтому в такой широкой локации их нет, и искать приходится в городах региона.

Не совпадает доля запросов без фильтров: 52% в валидации против 63% в тесте (59%, если не считать запросы по всему Авито). Мы эту долю не выравнивали. На оценку это расхождение, вероятно, почти не влияет: Recall@50 итогового решения на проверочных запросах без фильтров и с фильтрами одинаков — 0.950 в обоих случаях (`ranking/dense.ipynb`, раздел 8). Запросов по всему Авито в валидации только один, а в тесте их 222 (раздел 6).

Популярность знакомых текстов в тесте и в валидации:

In [14]:
val_seen = val.filter(pl.col("seen_text")).with_columns(popularity_bin(pl.col("text_tuples_in_fold")))
val_bins = val_seen.group_by("popularity_bin").len("val_texts").with_columns((pl.col("val_texts") / val_seen.height).alias("val_share"))

bins.join(bench_bins, on="popularity_bin", maintain_order="left").join(val_bins, on="popularity_bin", maintain_order="left").select(
    "в скольких запросах train встречается текст",
    pl.col("share").alias("доля в тесте"),
    pl.col("val_share").alias("доля в валидации"),
)

в скольких запросах train встречается текст,доля в тесте,доля в валидации
str,f64,f64
"""1""",0.278942,0.278882
"""2–3""",0.267916,0.268135
"""4–10""",0.281147,0.281032
"""11–50""",0.128997,0.128963
"""больше 50""",0.042999,0.042988


Доли совпадают почти точно, потому что знакомые тексты валидации так и набирались (раздел 3).

Сколько правильных объявлений у проверочного запроса:

In [15]:
val["relevant_item_ids"].list.len().value_counts().sort("relevant_item_ids").rename({
    "relevant_item_ids": "правильных объявлений у запроса",
    "count": "запросов",
})

правильных объявлений у запроса,запросов
u32,u32
1,4977
2,50
3,5


У 4 977 из 5 032 проверочных запросов одно правильное объявление, то есть у 99%. Во всём train запросов с несколькими выбранными объявлениями гораздо больше. Чтобы понять, на каком шаге отбора они исчезают, считаем их долю после каждого условия.

In [16]:
def several_items(df: pl.DataFrame) -> float:
    return (df["relevant_item_ids"].list.len() > 1).mean()


in_corpus = tuples.filter(pl.col("all_in_corpus"))

pl.DataFrame({
    "какие запросы": [
        "все запросы train",
        "подходящие: все выбранные объявления лежат в корпусе",
        "подходящие без фильтра «Автосервис, аренда», по одному запросу на текст",
        "валидация: знакомые тексты",
        "валидация: новые тексты",
        "валидация целиком",
    ],
    "запросов": [
        chosen_items.height,
        in_corpus.height,
        one_per_text.height,
        val.filter(pl.col("seen_text")).height,
        val.filter(~pl.col("seen_text")).height,
        val.height,
    ],
    "доля запросов с несколькими выбранными объявлениями": [
        (chosen_items["items"] > 1).mean(),
        several_items(in_corpus),
        several_items(one_per_text),
        several_items(val.filter(pl.col("seen_text"))),
        several_items(val.filter(~pl.col("seen_text"))),
        several_items(val),
    ],
})

какие запросы,запросов,доля запросов с несколькими выбранными объявлениями
str,i64,f64
"""все запросы train""",354463,0.14999
"""подходящие: все выбранные объявления лежат в корпусе""",18436,0.027663
"""подходящие без фильтра «Автосервис, аренда», по одному запросу на текст""",10542,0.017644
"""валидация: знакомые тексты""",1861,0.014508
"""валидация: новые тексты""",3171,0.00883
"""валидация целиком""",5032,0.01093


Главная причина — само условие «все выбранные объявления лежат в корпусе»: оно снижает долю запросов с несколькими выбранными объявлениями с 15% до 2.8%. Чем больше разных объявлений выбрали по запросу, тем меньше шанс, что все они лежат в корпусе. Удаление фильтра «Автосервис, аренда» и правило «один запрос на текст» снижают долю ещё до 1.8%. Наконец, знакомых текстов в валидации меньше, чем среди подходящих запросов, а у знакомых текстов несколько выборов бывает чаще: 1.5% против 0.9% у новых. В итоге в валидации таких запросов 1.1%.

Лежат ли правильные объявления в той же локации, где искали:

In [17]:
item_locations = ingested.load_items(["item_id", "item_location_id"])
relevant = val.select("query_id", "search_location_id", pl.col("relevant_item_ids").alias("item_id")).explode("item_id", empty_as_null=True)
relevant = relevant.join(item_locations, on="item_id")

pl.DataFrame({
    "показатель": ["доля правильных объявлений в той же локации, где искали"],
    "значение": [(relevant["item_location_id"] == relevant["search_location_id"]).mean()],
})

показатель,значение
str,f64
"""доля правильных объявлений в той же локации, где искали""",0.775923


78% правильных объявлений лежат в той же локации, где искали. Значит, локация поиска — сильная подсказка, но ограничиться ею нельзя: примерно каждое пятое правильное объявление лежит в другой локации, например в пригороде или в городе региона, по которому искали. Поэтому решение использует доли выборов по локациям, а не просто оставляет объявления из локации поиска.

## 5. Данные для обучения реранкера и ретривера

Как устроено решение, описано в начале ноутбука: ретривер отбирает около 600 кандидатов, реранкер заново упорядочивает их и берёт первые 50. Реранкер учится на запросах с известными ответами. Чтобы честно замерить его качество, это должны быть не проверочные запросы. Итоговую модель для ответа потом учим и на валидации, но её качество на валидации уже честно не измерить (`ranking/dense.ipynb`, раздел 5).

Кроме того, подсказки, которые получает реранкер, тоже получены из train: доли выборов по локациям считаются по строкам train, а эмбеддер, который оценивает похожесть запроса и объявления, доучивается на строках train. Если бы они строились по тем же строкам, на которых учится реранкер, они «знали» бы правильные ответы. Скажем, в строке train выбрали объявление из подмосковного города. Если эта строка учтена в долях выборов, доля этого города уже включает этот самый выбор, и на обучении подсказка «угадывает» чаще, чем сможет на тесте, где ответ заранее не учтён. Реранкер научился бы доверять таким подсказкам больше, чем они того стоят.

Поэтому из train без проверочных запросов берём ещё один набор — **обучение реранкера**. В него идут запросы, у которых хотя бы одно выбранное объявление лежит в корпусе; правильными считаются только объявления из корпуса. Условие мягче, чем у валидации (там в корпусе должны лежать все выбранные объявления), поэтому таких запросов больше. Тексты этих запросов не совпадают с текстами валидации, фильтра «Автосервис, аренда» у них нет, и, как в валидации, на каждый текст берётся один случайный запрос. Берутся все такие тексты, получается 6 487 запросов. Их строки тоже удаляются из train.

Всё, что осталось, — **обучение ретривера**. Ретривер здесь — всё, что готовит кандидатов. Сам BM25 ничему не учится, а доли выборов по локациям считаются и эмбеддер доучивается только по этой части. Размеры трёх частей:

In [18]:
ranker_queries = load_ranker_queries()
retriever_fold = load_retriever_fold()

pl.DataFrame({
    "часть train": ["валидация", "обучение реранкера", "обучение ретривера"],
    "запросов": [val.height, ranker_queries.height, retriever_fold.select(QUERY_COLUMNS).unique().height],
    "строк train": [train.height - train_fold.height, train_fold.height - retriever_fold.height, retriever_fold.height],
    "для чего": [
        "замер качества; итоговый реранкер для ответа учится и на ней",
        "обучение LightGBM",
        "доли выборов по локациям, дообучение эмбеддера",
    ],
})

часть train,запросов,строк train,для чего
str,i64,i64,str
"""валидация""",5032,5397,"""замер качества; итоговый реранкер для ответа учится и на ней"""
"""обучение реранкера""",6487,10115,"""обучение LightGBM"""
"""обучение ретривера""",342944,482161,"""доли выборов по локациям, дообучение эмбеддера"""


Вместе три части содержат все 497 673 строки train. Для обучения ретривера запросы не собираются: долям выборов и эмбеддеру нужны отдельные строки «поля поиска → выбранное объявление». Число разных наборов полей поиска в этой части показано только для сравнения.

**Проверка на утечки.** Утечкой называют случай, когда ответы проверочных запросов попали в данные для обучения: модель, видевшая ответ, покажет на проверке завышенное качество. Функции `check` и `check_ranker` останавливаются с ошибкой, если нарушено хотя бы одно условие.

`check` проверяет валидацию:

- все тексты проверочных запросов разные;
- все правильные объявления лежат в корпусе;
- пометка «текст встречается в train» совпадает с тем, что есть в train без проверочных запросов;
- в train не осталось ни одной строки с теми же полями поиска, что у проверочного запроса.

`check_ranker` проверяет обучение реранкера:

- тексты его запросов разные и не совпадают с текстами валидации;
- у каждого запроса есть хотя бы одно правильное объявление, и все правильные объявления лежат в корпусе;
- в обучении ретривера нет ни одной строки с теми же полями поиска, что у запроса валидации или обучения реранкера.

In [19]:
check(val, train_fold, corpus_item_ids)
check_ranker(ranker_queries, retriever_fold, val, corpus_item_ids)

print("проверки пройдены")

проверки пройдены


Запросы для обучения реранкера устроены не так, как тест. Сравниваем их с валидацией по доле знакомых текстов и доле запросов без фильтров:

In [20]:
pl.DataFrame({
    "что": ["валидация", "обучение реранкера"],
    "доля текстов, встречающихся в train": [val["seen_text"].mean(), ranker_queries["seen_text"].mean()],
    "доля без фильтров": [(val["search_infm_params_text"] == "").mean(), (ranker_queries["search_infm_params_text"] == "").mean()],
})

что,"доля текстов, встречающихся в train",доля без фильтров
str,f64,f64
"""валидация""",0.369833,0.52345
"""обучение реранкера""",0.968861,0.418684


В обучении реранкера 97% текстов знакомые, а в валидации — 37%: новые тексты с подходящими запросами уже забрала валидация. Это важно для реранкера, который опирается на подсказки эмбеддера. Эмбеддер доучивался на строках train и видел при обучении 89% текстов обучения реранкера, но только 32% текстов валидации и 28% текстов теста (`embeddings/models.ipynb`). На текстах, которые эмбеддер видел, его оценка похожести, вероятно, надёжнее. Тогда реранкер, обученный только на таких запросах, может доверять эмбеддеру больше, чем стоит для новых текстов (отдельно мы это не проверяли).

Поэтому в `ranking/dense.ipynb` (раздел 5) проверили, что даёт добавление в обучение реранкера половины проверочных запросов, считая качество на другой половине: Recall@50 вырос с 0.946 до 0.950. Часть прироста может давать и то, что обучающих запросов просто стало больше; этот опыт две причины не разделяет. Итоговый реранкер для ответа учится на запросах обучения реранкера и всей валидации.

## 6. Чего валидация не повторяет

Кроме доли запросов без фильтров, есть ещё три отличия от теста.

**Поиск по всему Авито.** 9% тестовых запросов (222) ищут не в разделе «Услуги», а по всему Авито. В train таких всего 34 строки (`ingestion/insights.ipynb`), и в валидацию попал только один такой запрос из 5 032 (таблица в разделе 4). Проверить качество на таких запросах у себя нельзя. Решение обрабатывает их так же, как остальные запросы.

**Запросы с несколькими правильными объявлениями.** По условию у тестового запроса подходящих объявлений «обычно одно-два, иногда больше». В валидации запросов с двумя и больше правильными объявлениями всего 1.1% (раздел 4), а во всём train — 15%: мы берём только запросы, у которых все выбранные объявления лежат в корпусе, а чем больше объявлений выбрали по запросу, тем реже все они там лежат. Запрос с двумя правильными объявлениями сложнее: чтобы получить за него 1, в 50 ответов должны попасть оба. Поэтому на тесте Recall@50, вероятно, ниже, чем на валидации; насколько — мы не проверяли.

**Точные повторы.** Строки проверочного запроса удаляются из train целиком, поэтому в валидации не бывает запросов, которые полностью (текст, локация, фильтры) повторяют запрос из train. В тесте такие запросы есть:

In [21]:
train_queries = train.select(QUERY_COLUMNS).unique()
repeated = bench.join(train_queries, on=QUERY_COLUMNS, how="semi")
repeated_rows = train.join(repeated.select(QUERY_COLUMNS), on=QUERY_COLUMNS, how="semi")

pl.DataFrame({
    "показатель": [
        "тестовых запросов, которые целиком повторяют запрос из train",
        "их доля среди тестовых запросов",
        "доля объявлений из корпуса среди выбранных по таким запросам в train",
        "доля объявлений из корпуса среди всех выбранных в train",
    ],
    "значение": [
        float(repeated.height),
        repeated.height / bench.height,
        repeated_rows.join(corpus_item_ids, on="item_id", how="semi").height / repeated_rows.height,
        train.join(corpus_item_ids, on="item_id", how="semi").height / train.height,
    ],
})

показатель,значение
str,f64
"""тестовых запросов, которые целиком повторяют запрос из train""",108.0
"""их доля среди тестовых запросов""",0.044046
"""доля объявлений из корпуса среди выбранных по таким запросам в train""",0.575221
"""доля объявлений из корпуса среди всех выбранных в train""",0.066329


108 тестовых запросов (4.4%) целиком повторяют запрос из train. По таким запросам в train выбирали объявления из корпуса в 58% случаев, а в среднем по train — только в 6.6%. У тестовых запросов все правильные объявления по условию лежат в корпусе. Поэтому, вероятно, в тесте по этим запросам часто выбирали те же объявления, что и в train. Проверить это нельзя: ответов теста у нас нет. Правило «вернуть то, что выбирали по этому же запросу в train» на тесте могло бы помочь, но на валидации его не проверить, поэтому в решение оно не входит.

## Итог

- Запрос — это набор полей поиска целиком. Строки train с одинаковыми полями склеиваются в один запрос, правильные ответы — все выбранные в них объявления.
- В валидации 5 032 запроса, каждый текст встречается один раз, все правильные объявления лежат в корпусе, и поиск идёт по всем 189 212 объявлениям, как в тесте. У 99% проверочных запросов одно правильное объявление.
- С тестом совпадают длина запросов, доля знакомых текстов (37%) и их популярность, доля запросов, в локации поиска которых в корпусе нет ни одного объявления (около 17%).
- Train поделён на три непересекающиеся части: валидация (5 032 запроса), обучение реранкера (6 487 запросов) и обучение ретривера (482 161 строка). Доли выборов по локациям считаются, а эмбеддер доучивается только на последней. Реранкер для замера качества учится на запросах обучения реранкера, итоговый реранкер для ответа — ещё и на валидации.
- Чего валидация не повторяет: запросов без фильтров в ней меньше (52% против 63%), запросов по всему Авито почти нет (1 из 5 032 против 9% теста), запросов с несколькими правильными объявлениями мало (1.1%, а в тесте их, судя по условию, больше), и нет точных повторов запросов из train (4.4% теста).
- Лучшая оценка итогового решения на валидации — 0.950, на лидерборде оно дало 0.922 (`ranking/dense.ipynb`). Вероятно, часть этой разницы объясняется отличиями валидации от теста, но мы это не проверяли.